# 🤖 Document AI Prototype — Generic Multi-PDF Table Extraction

**Dual-runtime notebook**: jalankan di Google Colab untuk prototyping/one-off processing.

**Alur**: Install SDK → Auth GCP → Upload modul → Proses PDF → Export Excel/CSV → Download

## Cell 1: Install Dependencies

In [ ]:
!pip install -q google-cloud-documentai google-cloud-storage pypdf pandas openpyxl

## Cell 2: Auth GCP

Pilih salah satu metode:
- **A. Interactive auth** (akun Google Anda)
- **B. Service Account JSON** (upload file kunci)

In [ ]:
# Metode A: Interactive auth (akun Google Anda)
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

# ATAU Metode B: Service Account JSON
# from google.colab import files
# uploaded = files.upload()  # upload service_account.json
# import os
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = list(uploaded.keys())[0]

## Cell 3: Konfigurasi GCP

Isi `PROJECT_ID`, `LOCATION`, dan `PROCESSOR_ID` sesuai processor Document AI Anda.

In [ ]:
import os

# ==== KONFIGURASI GCP ====
os.environ['GCP_PROJECT_ID'] = 'YOUR_PROJECT_ID'      # Ganti dengan project ID Anda
os.environ['GCP_LOCATION'] = 'us'                     # 'us' atau 'eu'
os.environ['DOCAI_PROCESSOR_ID'] = 'YOUR_PROCESSOR_ID'  # Ganti dengan processor ID Anda

print("Config set:")
print(f"  Project: {os.environ['GCP_PROJECT_ID']}")
print(f"  Location: {os.environ['GCP_LOCATION']}")
print(f"  Processor: {os.environ['DOCAI_PROCESSOR_ID']}")

## Cell 4: Upload Modul Document AI

Upload file dari `core/mcp-unified/services/documentai/`:
- `config.py`
- `engine.py`
- `pdf_handler.py`
- `text_cleaner.py`
- `exporter.py`

In [ ]:
from google.colab import files
import os

# Buat folder untuk modul
os.makedirs('/content/documentai', exist_ok=True)

# Upload modul (pilih file dari services/documentai/)
print("Upload file berikut satu per satu:")
print("  - config.py")
print("  - engine.py")
print("  - pdf_handler.py")
print("  - text_cleaner.py")
print("  - exporter.py")
uploaded = files.upload()

# Pindahkan ke folder documentai
for filename in uploaded.keys():
    os.rename(filename, f'/content/documentai/{filename}')

print("\nModul tersimpan di /content/documentai/")
print(os.listdir('/content/documentai'))

## Cell 5: Upload PDF yang Akan Diproses

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data/input', exist_ok=True)
os.makedirs('/content/data/output', exist_ok=True)

print("Upload file PDF yang akan diproses:")
uploaded_pdf = files.upload()

pdf_files = []
for filename in uploaded_pdf.keys():
    os.rename(filename, f'/content/data/input/{filename}')
    pdf_files.append(f'/content/data/input/{filename}')

print(f"\nPDF tersedia: {pdf_files}")

## Cell 6: Proses PDF — Ekstraksi Tabel

Ganti `PDF_PATH` dan `PAGE_RANGE` sesuai kebutuhan.

In [ ]:
import sys
sys.path.insert(0, '/content/documentai')

from engine import DocumentAIEngine
from exporter import export_both

# ==== KONFIGURASI PROSES ====
PDF_PATH = '/content/data/input/Surat BPJS-0726.pdf'  # Ganti dengan nama file Anda
PAGE_RANGE = '4-7'  # 'all', '4-7', '1,3,5', atau '1'
OUTPUT_FORMAT = 'both'  # 'excel', 'csv', atau 'both'

# Inisialisasi engine
engine = DocumentAIEngine()

# Ekstraksi tabel
print(f"Memproses: {PDF_PATH}")
print(f"Page range: {PAGE_RANGE}")
tables = engine.extract_tables(PDF_PATH, PAGE_RANGE)

print(f"\nDitemukan {len(tables)} tabel:")
for i, table in enumerate(tables):
    print(f"  Table {i+1}: {len(table['rows'])} baris, {len(table['headers'])} kolom")
    print(f"    Headers: {table['headers']}")
    print(f"    Pages: {table['page_numbers']}")

# Preview 3 baris pertama
if tables:
    print("\nPreview (3 baris pertama):")
    for row in tables[0]['rows'][:3]:
        print(f"  {row}")

## Cell 7: Export ke Excel/CSV

In [ ]:
import os
from pathlib import Path

# Tentukan nama output berdasarkan file input
input_name = Path(PDF_PATH).stem
base_output = f'/content/data/output/{input_name}_extracted'

# Export
if OUTPUT_FORMAT == 'both':
    result = export_both(tables, base_output)
    print(f"Excel: {result['excel_path']}")
    print(f"CSV: {result['csv_paths']}")
elif OUTPUT_FORMAT == 'excel':
    from exporter import export_to_excel
    path = export_to_excel(tables, f'{base_output}.xlsx')
    print(f"Excel: {path}")
else:
    from exporter import export_to_csv
    paths = export_to_csv(tables, f'{base_output}.csv')
    print(f"CSV: {paths}")

print("\nFile output:")
for f in os.listdir('/content/data/output'):
    print(f"  {f}")

## Cell 8: Download Hasil

In [ ]:
from google.colab import files
import os

output_dir = '/content/data/output'
for f in os.listdir(output_dir):
    file_path = os.path.join(output_dir, f)
    if os.path.isfile(file_path):
        print(f"Downloading: {f}")
        files.download(file_path)